# 02 · Speed team: smaller and faster, same accuracy

The model has to run on a laptop CPU (and one day on a wristband). Your job is to make
it **faster** and **smaller** without losing accuracy.

**Challenge rule:** the fastest model by `speed_report.py` wins, as long as its mean
CLER is within **X points** of the baseline (X announced at 10:30). Latency = median
ms to process **1 second of EMG**, batch 1, **1 CPU thread**.

We start with the classic first move: **INT8 dynamic quantization**. LSTM and Linear
weights go from 32-bit floats to 8-bit integers, about 3.6x smaller. Whether it's
*faster* depends on your CPU. On Intel/AMD (and Colab), PyTorch uses the `fbgemm`
backend: in our test on Colab, INT8 was **2.4x faster** (145 -> 60 ms). On Apple Silicon
it uses `qnnpack`, and on an M5 Pro the INT8 LSTM was **8x slower** than fp32
(12.5 -> 99 ms). Measure; don't assume.

*Model: Meta, generic-neuromotor-interface (CC BY-NC 4.0).*

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# Local: you've already run `pip install -r requirements.txt` in this repo, so skip ahead.
# Colab: this cell clones the repo, installs it, and points at the shared data.
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/sandeepvijayarao09/wearable-ai-build-day.git"  # organizers fill in
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    %cd /content/repo
    !pip install -q -r requirements.txt
    # Shared data: in Drive, right-click the shared "emg_data" folder ->
    # Organize -> Add shortcut -> My Drive. Then:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["EMG_DATA_DIR"] = "/content/drive/MyDrive/emg_data"
    os.environ["EMG_MODELS_DIR"] = "/content/drive/MyDrive/emg_data/emg_models"
else:
    # Run from the repo root so `emg_kit` imports.
    if not os.path.exists("emg_kit") and os.path.exists("../emg_kit"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from emg_kit.core import (benchmark_ms, evaluate, load_model, model_size_mb,
                          quantize_int8, save_whole_model)

fp32 = load_model()
int8 = quantize_int8(fp32)
print(int8)   # LSTM and Linear are now Dynamic Quantized modules

## 1. Size and latency (fast, ~1 minute)

In [ ]:
rows = []
for name, m in [("baseline fp32", fp32), ("int8 dynamic", int8)]:
    rows.append({"model": name,
                 "size_MB": round(model_size_mb(m), 1),
                 "ms_per_prediction": round(benchmark_ms(m), 2)})
speed = pd.DataFrame(rows).set_index("model")
speed["speedup"] = (speed.loc["baseline fp32", "ms_per_prediction"]
                    / speed["ms_per_prediction"]).round(2)
speed

## 2. Did accuracy survive? (~30 s per user per model on a laptop, ~5 min on Colab)

In [ ]:
acc = {}
for name, m in [("baseline fp32", fp32), ("int8 dynamic", int8)]:
    print(f"--- {name}")
    acc[name] = evaluate(m)
speed["mean_CLER"] = [acc[k].cler.mean() for k in speed.index]
speed["CLER_points_vs_baseline"] = (
    (speed.mean_CLER - speed.loc["baseline fp32", "mean_CLER"]) * 100).round(2)
speed

In [ ]:
save_whole_model(int8, "models/int8_dynamic.pt")
print("saved models/int8_dynamic.pt -- score it with:")
print("  python speed_report.py --model models/int8_dynamic.pt")

## 3. Your turn: ideas to try

Each idea should end with `save_whole_model(m, "models/<name>.pt")`, then score it with
`speed_report.py`.

1. **Threads.** `benchmark_ms(m, threads=4)`. Faster, but the official rule is 1 thread. Why do you think that is?
2. **`torch.compile` / TorchScript.** `torch.jit.script(fp32)` or `torch.jit.trace`. Does the LSTM get faster?
3. **Prune the LSTM.** `torch.nn.utils.prune.l1_unstructured` on LSTM weights, then quantize.
4. **Smaller window = lower latency?** `benchmark_ms(m, window_s=0.25)`. What does a real-time system actually need?
5. **Half precision** (`m.half()`). It's usually *slower* on CPU. Measure it and explain why.
6. **Static quantization of the Conv1d** (harder). Dynamic quantization skipped the conv layer.

Note: retraining isn't feasible in one day on a CPU. Stick to post-training tricks.